[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/minimal_surfaces/minimal_surfaces.ipynb)

# Minimal Surfaces and Harmonic Conjugates

This example computes minimal surfaces, the shapes of soap films, whose area no small change of their interior can reduce. Turning every small piece of a minimal surface a quarter turn in its own tangent plane fits the pieces together into another minimal surface, its harmonic conjugate, and turning them by any other angle bends the one into the other without stretching, along Bonnet's family.

A quarter turn in a triangle's plane is the commutator with that plane, `sides.commutator(normals.dual())`, and that one expression is both the area gradient and the differential the conjugate integrates: summed at the vertices, the turned sides are the cotangent Laplacian's pull, `laplacian * vertices`. The conjugate needs boundary sums alone, since neighbouring triangles run their shared edge in opposite senses and the interior cancels. Bonnet's family turns every side within its own tangent plane, and so keeps the normals and the first fundamental form. The method follows [Pinkall and Polthier (1993)](https://doi.org/10.1080/10586458.1993.10504266).

In the vector notation of discrete differential geometry, the area gradient at a triangle's corner is $\nabla_{p_i} A_T = \frac{1}{2} N_T \times e_i$, the cotangent Laplacian is $(L X)_i = \frac{1}{2}\sum_{j} (\cot \alpha_{ij} + \cot \beta_{ij})(X_i - X_j) = -2 H_i A_i N_i$, the conjugate integrates $dX^* = N \times dX$, and Bonnet's family is $X_\theta = \cos\theta\, X + \sin\theta\, X^*$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import stack
from numga.algebras import VGA3D as ga
from numga.sparse import spdiag
from examples import instantiate
from examples.surfaces.minimal_surfaces import render, scenarios
from examples.util.animation import save_figure, show_animation

mesh_module = instantiate("examples.util.mesh", ga)
Mesh = mesh_module.Mesh
Scalar = ga.gatype.scalar()                                # Scalar+
Point = ga.gatype.antivector()                             # Antivector-
# Mean curvature, measured against the triangles' own winding.
Curvature = ga.gatype.antipseudoscalar()                   # Antipseudoscalar-

np.set_printoptions(precision=4, suppress=True)

## Area gradient and soap-film relaxation

A soap film is where the area gradient vanishes inside its wire, here a circle raised into four arches and spanned at first by a dome. Each damped step moves the interior towards the harmonic surface of the current triangles, where their pull vanishes, and the area falls at every step.

In [ ]:
disk_radius = 1.0
n_disk_divisions = 10
arch_height = 0.35
dome_height = 0.55
relaxation_step_size = 0.14
n_relaxation_steps = 40
frame_duration_ms = 40

In [ ]:
plateau_mesh, flat_vertices, initial_vertices = scenarios.four_arch_dome(
    disk_radius, n_disk_divisions, arch_height, dome_height,
)                                                          # Mesh, Point-[vertices], Point-[vertices]
corners = plateau_mesh.incidence[2, 0]                     # [corners] Scalar+[faces <- vertices]
# The side facing each corner, run the way its triangle winds.
sides = plateau_mesh.boundary[2, 1] * (plateau_mesh.d[0] * initial_vertices)  # [corners] Point-[faces]
patches = (sides[0] & sides[1]) / 2                        # Patch-[faces]
areas = patches.norm()                                     # Scalar+[faces]
normals = patches / areas                                  # Patch-[faces]
# Each corner's hat rises from zero along the facing side to one at the corner.
corner_gradients = (sides | patches.inverse()) / 2         # [corners] Plane+[faces]
# A quarter turn in each triangle's plane makes each facing side twice its corner's area gradient.
turned_sides = sides.commutator(normals.dual())            # [corners] Point-[faces]
area_pull = (~corners * turned_sides).sum(axis=0) / 2      # Point-[vertices]

# The cotangent Laplacian: each triangle's corner gradients paired with one another over its area, pulled back to
# the vertices through both corners of each pair. Its pull on the dome is the area pull.
local_laplacian = (corner_gradients[:, None] | corner_gradients[None, :]) * areas  # [corners, corners] Scalar+[faces]
laplacian = (~corners[:, None] * spdiag(local_laplacian) * corners[None, :]).sum(axis=0).sum(axis=0)  # Scalar+[vertices <- vertices]

In [ ]:
def relax_plateau(
    mesh: Mesh, start_vertices: Point, step_size: float, n_steps: int,
) -> Iterator[tuple[Point, Scalar, Curvature]]:
    """The surface at each damped harmonic step towards least area, its boundary wire held, with its total area and
    its mean curvature at the vertices, zero on the wire."""
    corners = mesh.incidence[2, 0]                         # [corners] Scalar+[faces <- vertices]
    interior = mesh.reading(mesh.interior(0))              # Scalar+[interior <- vertices]
    exterior = mesh.reading(mesh.exterior(0))              # Scalar+[exterior <- vertices]
    wire = exterior * start_vertices                       # Point-[exterior]
    vertices = start_vertices                              # Point-[vertices]
    for _ in range(n_steps + 1):
        sides = mesh.boundary[2, 1] * (mesh.d[0] * vertices)  # [corners] Point-[faces]
        patches = (sides[0] & sides[1]) / 2                # Patch-[faces]
        areas = patches.norm()                             # Scalar+[faces]
        normals = patches / areas                          # Patch-[faces]
        corner_gradients = (sides | patches.inverse()) / 2  # [corners] Plane+[faces]
        local_laplacian = (corner_gradients[:, None] | corner_gradients[None, :]) * areas  # [corners, corners] Scalar+[faces]
        laplacian = (~corners[:, None] * spdiag(local_laplacian) * corners[None, :]).sum(axis=0).sum(axis=0)  # Scalar+[vertices <- vertices]
        turned_sides = sides.commutator(normals.dual())    # [corners] Point-[faces]
        area_pull = interior * (~corners * turned_sides).sum(axis=0) / 2  # Point-[interior]
        # A vertex's area pull is twice its mean curvature times its third of the patches around it.
        vertex_patches = interior * (~corners.mean(axis=0) * patches)  # Patch-[interior]
        mean_curvature = (area_pull & vertex_patches.inverse()) / 2  # Curvature-[interior]
        yield vertices, areas.sites.sum(), ~interior * mean_curvature
        # The harmonic surface of this step's triangles: the interior where their pull vanishes, the wire held.
        wire_pull = (interior * laplacian * ~exterior) * wire  # Point-[interior]
        harmonic_interior = ((interior * laplacian * ~interior) * Point).solve(-wire_pull)  # Point-[interior]
        harmonic = ~interior * harmonic_interior + ~exterior * wire  # Point-[vertices]
        # Moving part of the way keeps the triangles, and with them the Laplacian, from changing too fast.
        vertices = (1 - step_size) * vertices + step_size * harmonic  # Point-[vertices]


relaxation_states = list(relax_plateau(
    plateau_mesh, initial_vertices, relaxation_step_size, n_relaxation_steps,
))
show_animation(
    render.animate_plateau(plateau_mesh, flat_vertices, relaxation_states),
    "minimal_surfaces_plateau",
    frame_duration_ms,
    n_frames=n_relaxation_steps + 1,
    forward_then_back=True,
)

The coral tint marks the magnitude of the interior mean curvature, which fades as the domed sheet sinks onto the saddle-shaped soap film spanning the four-arch wire. The teal rings and coloured radial spokes follow fixed curves in the planar disk domain.


In [ ]:
relaxation_areas = stack([area for _, area, _ in relaxation_states])  # [steps + 1] Scalar+
_, _, initial_curvature = relaxation_states[0]
_, _, final_curvature = relaxation_states[-1]
print("Surface area, initial and minimal:", relaxation_areas[[0, -1]].to_array())
print("Largest mean curvature, initial and minimal:",
      np.abs(initial_curvature.batch().to_array()).max(), np.abs(final_curvature.batch().to_array()).max())

## Conjugates from boundary sums

The conjugate is the surface whose cotangent Laplacian's pull is half the sum of the sides facing each vertex, `laplacian * conjugate == boundary_flux`, up to a translation. Enneper's surface joins the relaxed film because its conjugate is known in closed form.

In [ ]:
enneper_radius = 0.8
n_enneper_divisions = 10

In [ ]:
def conjugate_minimal_surface(mesh: Mesh, vertices: Point) -> tuple[Point, Point]:
    """The harmonic conjugate of a minimal surface, up to a translation, and the sums of the sides facing each vertex
    that it is solved from."""
    corners = mesh.incidence[2, 0]                         # [corners] Scalar+[faces <- vertices]
    sides = mesh.boundary[2, 1] * (mesh.d[0] * vertices)   # [corners] Point-[faces]
    patches = (sides[0] & sides[1]) / 2                    # Patch-[faces]
    areas = patches.norm()                                 # Scalar+[faces]
    corner_gradients = (sides | patches.inverse()) / 2     # [corners] Plane+[faces]
    local_laplacian = (corner_gradients[:, None] | corner_gradients[None, :]) * areas  # [corners, corners] Scalar+[faces]
    laplacian = (~corners[:, None] * spdiag(local_laplacian) * corners[None, :]).sum(axis=0).sum(axis=0)  # Scalar+[vertices <- vertices]
    # The corner gradients read the quarter-turned differential over each triangle as half the bare facing side:
    # the triangle's area and its normal cancel.
    boundary_flux = (~corners * sides).sum(axis=0) / 2     # Point-[vertices]
    # The Laplacian leaves translations free, and the least-squares solve picks one conjugate among them.
    conjugate = (laplacian * Point).lstsq(boundary_flux)   # Point-[vertices]
    return conjugate, boundary_flux


minimal_vertices, _, _ = relaxation_states[-1]
plateau_conjugate, plateau_flux = conjugate_minimal_surface(plateau_mesh, minimal_vertices)  # Point-[vertices], Point-[vertices]
enneper_mesh, enneper_flat, enneper_vertices, exact_enneper_conjugate = scenarios.enneper_disk(
    enneper_radius, n_enneper_divisions,
)                                                          # Mesh, Point-[vertices], Point-[vertices], Point-[vertices]
enneper_conjugate, enneper_flux = conjugate_minimal_surface(enneper_mesh, enneper_vertices)  # Point-[vertices], Point-[vertices]

save_figure(
    render.draw_conjugates(
        plateau_mesh, flat_vertices, minimal_vertices, plateau_conjugate,
        enneper_mesh, enneper_flat, enneper_vertices, enneper_conjugate,
    ),
    "minimal_surfaces_conjugates",
);

The top row shows the relaxed soap film on the left and its conjugate on the right, where the four arches turn into four diagonally shifted lobes; the bottom row shows Enneper's surface and its conjugate, Enneper's surface turned by half a right angle in space. Matched domain rings and radial spokes cross at right angles between each surface and its conjugate.


## Bonnet's family from catenoid to helicoid

Bonnet's family, `vertices * np.cos(angle) + conjugate * np.sin(angle)`, bends a catenoid into its conjugate, a helicoid, through minimal surfaces with the same normals, lengths and angles. The helicoid rises as it turns and would not close around the catenoid's neck, so the catenoid is slit along a meridian.

In [ ]:
catenoid_height = 1.15
catenoid_azimuth_span = 0.85 * np.pi
n_catenoid_heights = 22
n_catenoid_azimuths = 44
n_bonnet_frames = 36

In [ ]:
def bonnet_family(
    mesh: Mesh, vertices: Point, conjugate: Point, angles: np.ndarray,
) -> Iterator[tuple[Point, Point]]:
    """The minimal surfaces of Bonnet's family between a minimal surface and its harmonic conjugate, one at each
    angle, each with its unit normals at the vertices, the patches of the faces around each vertex summed."""
    corners = mesh.incidence[2, 0]
    for angle in angles:
        surface = vertices * np.cos(angle) + conjugate * np.sin(angle)  # Point-[vertices]
        sides = mesh.boundary[2, 1] * (mesh.d[0] * surface)            # [corners] Point-[faces]
        normals = (~corners * (sides[0] & sides[1])).sum(axis=0).normalized()   # Patch-[vertices]
        yield surface, normals


catenoid_mesh, catenoid_vertices, exact_helicoid, catenoid_rings, catenoid_meridians = scenarios.slit_catenoid(
    catenoid_height, catenoid_azimuth_span, n_catenoid_heights, n_catenoid_azimuths,
)                                                          # Mesh, Point-[vertices], Point-[vertices], [heights, azimuths], [azimuths, heights]
helicoid_vertices, catenoid_flux = conjugate_minimal_surface(catenoid_mesh, catenoid_vertices)  # Point-[vertices], Point-[vertices]
bonnet_angles = np.linspace(0.0, np.pi / 2, n_bonnet_frames)
show_animation(
    render.animate_bonnet(
        catenoid_mesh, catenoid_rings, catenoid_meridians,
        bonnet_family(catenoid_mesh, catenoid_vertices, helicoid_vertices, bonnet_angles),
    ),
    "minimal_surfaces_bonnet",
    frame_duration_ms,
    n_frames=n_bonnet_frames,
    forward_then_back=True,
)

The teal rings and coral meridians of the slit catenoid slide into the helices and rulings of the helicoid without changing their lengths or the angles at which they cross. The gold pins are the normals, which move with their vertices and keep their directions throughout.

In [ ]:
# checks
# Each turned side is twice its corner's area gradient, read as a direction, and summed at the vertices the turned
# sides are the area gradient and the cotangent Laplacian's pull alike.
np.testing.assert_allclose((turned_sides - 2 * areas * corner_gradients.dual()).kernel, 0.0, atol=1e-13)
area_gradient = (~corners * (areas * corner_gradients)).sum(axis=0)  # Plane+[vertices]
np.testing.assert_allclose((area_pull - area_gradient.dual()).kernel, 0.0, atol=1e-13)
np.testing.assert_allclose((area_pull - laplacian * initial_vertices).kernel, 0.0, atol=1e-12)

# The film's area falls at every step.
assert np.all(relaxation_areas[1:] < relaxation_areas[:-1])

# The sides facing each vertex cancel in the interior.
np.testing.assert_allclose((plateau_mesh.reading(plateau_mesh.interior(0)) * plateau_flux).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((enneper_mesh.reading(enneper_mesh.interior(0)) * enneper_flux).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((catenoid_mesh.reading(catenoid_mesh.interior(0)) * catenoid_flux).kernel, 0.0, atol=1e-14)

# The recovered conjugates match the exact ones up to a translation.
centred_enneper = (enneper_conjugate.batch() - enneper_conjugate.batch().mean(axis=0)).field()
centred_exact_enneper = (exact_enneper_conjugate.batch() - exact_enneper_conjugate.batch().mean(axis=0)).field()
np.testing.assert_allclose(centred_enneper.kernel, centred_exact_enneper.kernel, atol=2e-2)
centred_helicoid = (helicoid_vertices.batch() - helicoid_vertices.batch().mean(axis=0)).field()
centred_exact_helicoid = (exact_helicoid.batch() - exact_helicoid.batch().mean(axis=0)).field()
np.testing.assert_allclose(centred_helicoid.kernel, centred_exact_helicoid.kernel, atol=2e-2)